# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [1]:
%pip install -q duckdb huggingface_hub pandas

import os
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")

In [3]:
grain_check = con.sql("""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as cnt
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
""").df()

print("Duplicate grain rows found:", len(grain_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate grain rows found: 6390


In [4]:
duplicate_examples = con.sql("""
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
    WHERE (client_hash_id, content_hash_id, report_date) IN (
        SELECT client_hash_id, content_hash_id, report_date
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
        GROUP BY client_hash_id, content_hash_id, report_date
        HAVING COUNT(*) > 1
    )
    ORDER BY client_hash_id, content_hash_id, report_date
    LIMIT 10
""").df()

duplicate_examples

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,report_date,client_hash_id,content_hash_id,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,gsc_impressions,gsc_clicks,gsc_sum_position,...,sessions_ai,ai_chatgpt,ai_perplexity,ai_gemini,ai_copilot,ai_claude,ai_meta,ai_other,scroll_events,month
0,2026-06-13,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
1,2026-06-13,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
2,2026-06-14,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
3,2026-06-14,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
4,2026-06-15,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
5,2026-06-15,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
6,2026-06-16,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
7,2026-06-16,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
8,2026-06-17,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06
9,2026-06-17,client_06d356715a8ff3b6,content_01dd034a486dd411,True,True,False,False,0,0,0,...,0,0,0,0,0,0,0,0,0,2026-06


In [7]:
# Take just ONE duplicate pair to inspect closely
one_pair = con.sql("""
    SELECT *
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
    WHERE client_hash_id = 'client_06d356715a8ff3b6'
      AND content_hash_id = 'content_01dd034a486dd411'
      AND report_date = '2026-06-13'
""").df()

print("Number of rows in this exact pair:", len(one_pair))

# Compare row 0 and row 1 column by column
row0 = one_pair.iloc[0]
row1 = one_pair.iloc[1]

for col in one_pair.columns:
    if row0[col] != row1[col]:
        print(f"DIFFERENT column: {col}  |  row0 = {row0[col]}  |  row1 = {row1[col]}")

Number of rows in this exact pair: 2
DIFFERENT column: gsc_avg_position  |  row0 = nan  |  row1 = nan


In [8]:
import pandas as pd

for col in one_pair.columns:
    val0 = row0[col]
    val1 = row1[col]
    # Skip the false-alarm case where both are just missing (NaN)
    if pd.isna(val0) and pd.isna(val1):
        continue
    if val0 != val1:
        print(f"DIFFERENT column: {col}  |  row0 = {val0}  |  row1 = {val1}")

In [9]:
# How many rows are exact full-row duplicates (not just same client/page/date)?
exact_dupes = con.sql("""
    SELECT COUNT(*) as extra_duplicate_rows
    FROM (
        SELECT *, COUNT(*) as cnt
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
        GROUP BY ALL
        HAVING COUNT(*) > 1
    )
""").df()

print(exact_dupes)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   extra_duplicate_rows
0                  6390


One row represents one content page's performance on one specific calendar day, for one client. When verifying this, I found 6,390 groups of client+page+date combinations with more than one row. Investigating directly, I confirmed these are exact, fully identical duplicate rows across all 31 columns — not a hidden extra dimension I was missing, but a genuine data quality issue in the source (likely a duplicate-write during pipeline loading). My original grain definition is correct; the practical fix is de-duplicating rows before any analysis or modeling.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [10]:
clean_check = con.sql("""
    SELECT COUNT(*) as remaining_duplicates
    FROM (
        SELECT DISTINCT *
        FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
    ) t
    GROUP BY t.client_hash_id, t.content_hash_id, t.report_date
    HAVING COUNT(*) > 1
""").df()

print("Duplicates remaining after SELECT DISTINCT:", len(clean_check))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicates remaining after SELECT DISTINCT: 0


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [11]:
counts = con.sql("""
    SELECT COUNT(*) as total_rows,
           COUNT(DISTINCT client_hash_id) as unique_clients,
           COUNT(DISTINCT content_hash_id) as unique_content
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
""").df()
print(counts)

   total_rows  unique_clients  unique_content
0    11694072              65          409205


In [12]:
missing = con.sql("""
    SELECT
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) as missing_impressions,
        SUM(CASE WHEN ga4_data_available = False THEN 1 ELSE 0 END) as ga4_unavailable_rows,
        COUNT(*) as total
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
""").df()
print(missing)

   missing_impressions  ga4_unavailable_rows     total
0                  0.0             8651918.0  11694072


In [13]:
per_client = con.sql("""
    SELECT client_hash_id, MIN(report_date) as first_date, MAX(report_date) as last_date, COUNT(*) as row_count
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance_sample.parquet')
    GROUP BY client_hash_id
    ORDER BY row_count ASC
    LIMIT 10
""").df()
print(per_client)

            client_hash_id first_date  last_date  row_count
0  client_2c32078d69f2cbad 2026-06-01 2026-06-25        325
1  client_a1203ffecad62470 2026-06-01 2026-06-30       2250
2  client_59256b0571e0c970 2026-06-01 2026-06-30       3330
3  client_0e1acc6cd57b0eba 2026-06-01 2026-06-30       4410
4  client_80ee5b7bd5f4eb89 2026-06-01 2026-06-30       5880
5  client_770e8e5faa9cddfe 2026-06-01 2026-06-30       6000
6  client_0797ff3a1fc9a6a5 2026-06-01 2026-06-25       6500
7  client_8ae2bfb5aa1ffa1e 2026-06-01 2026-06-30      10710
8  client_08d2847f24cf89c1 2026-06-01 2026-06-30      12060
9  client_ccdd78843409c8c7 2026-06-01 2026-06-30      12510


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

The raw data contains exact duplicate rows for roughly 6,390 client/page/day combinations — likely from a pipeline loading issue. Any analysis or model built on this data must first de-duplicate rows (e.g., using SELECT DISTINCT), or these repeats will silently inflate counts and skew results.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.